# Anomaly & Efficiency Detection

This notebook detects anomalies and efficiency issues in the Green Supply Chain Tracker system, helping identify unusual patterns, inefficiencies, and optimization opportunities.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")


## 1. Load and Prepare Data


In [ ]:
# Generate comprehensive shipment data with anomalies
np.random.seed(42)

vehicle_types = ['truck', 'train', 'ship', 'plane', 'van']
fuel_types = ['diesel', 'petrol', 'electric', 'hybrid', 'cng']

emission_factors = {
    ('truck', 'diesel'): 0.62, ('truck', 'petrol'): 0.68, ('truck', 'electric'): 0.15,
    ('truck', 'hybrid'): 0.35, ('truck', 'cng'): 0.45,
    ('van', 'diesel'): 0.48, ('van', 'petrol'): 0.52, ('van', 'electric'): 0.12,
    ('train', 'diesel'): 0.22, ('train', 'electric'): 0.05,
    ('ship', 'diesel'): 0.015, ('plane', 'petrol'): 1.2,
}

n_samples = 500
data = []
base_date = datetime(2024, 1, 1)

for i in range(n_samples):
    vehicle = np.random.choice(vehicle_types)
    
    if vehicle == 'truck':
        fuel = np.random.choice(['diesel', 'petrol', 'electric', 'hybrid', 'cng'])
    elif vehicle == 'van':
        fuel = np.random.choice(['diesel', 'petrol', 'electric'])
    elif vehicle == 'train':
        fuel = np.random.choice(['diesel', 'electric'])
    elif vehicle == 'ship':
        fuel = 'diesel'
    else:
        fuel = 'petrol'
    
    weight = np.random.uniform(100, 5000)
    distance = np.random.uniform(50, 2000)
    
    # Introduce anomalies (5% of data)
    is_anomaly = np.random.random() < 0.05
    
    if is_anomaly:
        # Anomaly types: excessive emissions, unusual route, inefficient vehicle choice
        anomaly_type = np.random.choice(['high_emission', 'inefficient_route', 'wrong_vehicle'])
        
        if anomaly_type == 'high_emission':
            # Emissions 3x higher than expected
            weight *= 3
        elif anomaly_type == 'inefficient_route':
            # Route 2x longer than necessary
            distance *= 2
        else:  # wrong_vehicle
            # Using inefficient vehicle for the route
            if distance < 200:
                vehicle = 'plane'  # Using plane for short distance
            else:
                vehicle = 'van'  # Using van for long distance with heavy cargo
                weight = np.random.uniform(4000, 5000)
    
    factor = emission_factors.get((vehicle, fuel), 0.5)
    base_emissions = factor * distance * (weight / 1000)
    noise = np.random.normal(0, base_emissions * 0.1)
    emissions = max(0, base_emissions + noise)
    
    # Calculate efficiency metrics
    emissions_per_km = emissions / distance if distance > 0 else 0
    emissions_per_kg = emissions / weight if weight > 0 else 0
    eco_points = max(0, round(1000 - emissions_per_km * 10))
    
    # Delivery time (simulated)
    speed = 20  # km/h
    delivery_hours = distance / speed
    delivery_time = base_date + timedelta(days=np.random.randint(0, 90), hours=delivery_hours)
    
    data.append({
        'id': f'SH{i:04d}',
        'vehicle_type': vehicle,
        'fuel_type': fuel,
        'weight': weight,
        'distance': distance,
        'emissions_co2': emissions,
        'emissions_per_km': emissions_per_km,
        'emissions_per_kg': emissions_per_kg,
        'eco_points': eco_points,
        'delivery_time': delivery_time,
        'is_anomaly': is_anomaly
    })

df = pd.DataFrame(data)
df['delivery_time'] = pd.to_datetime(df['delivery_time'])

print(f"Dataset shape: {df.shape}")
print(f"\nAnomalies detected: {df['is_anomaly'].sum()} ({df['is_anomaly'].mean()*100:.1f}%)")
print("\nFirst few rows:")
df.head()


## 2. Statistical Anomaly Detection


In [ ]:
# Z-score based anomaly detection
def detect_zscore_anomalies(df, columns, threshold=3):
    """Detect anomalies using Z-score method"""
    anomalies = pd.DataFrame()
    
    for col in columns:
        z_scores = np.abs(stats.zscore(df[col]))
        col_anomalies = df[z_scores > threshold].copy()
        col_anomalies['anomaly_type'] = f'high_{col}'
        col_anomalies['z_score'] = z_scores[z_scores > threshold]
        anomalies = pd.concat([anomalies, col_anomalies])
    
    return anomalies.drop_duplicates(subset=['id'])

# Detect anomalies in key metrics
anomaly_columns = ['emissions_co2', 'emissions_per_km', 'distance', 'weight']
zscore_anomalies = detect_zscore_anomalies(df, anomaly_columns, threshold=2.5)

print(f"Z-score anomalies detected: {len(zscore_anomalies)}")
print("\nAnomaly types distribution:")
print(zscore_anomalies['anomaly_type'].value_counts())


In [ ]:
# Visualize anomalies
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Emissions distribution with anomalies
axes[0, 0].hist(df['emissions_co2'], bins=50, alpha=0.7, label='Normal', color='lightblue')
if len(zscore_anomalies) > 0:
    axes[0, 0].hist(zscore_anomalies['emissions_co2'], bins=20, alpha=0.8, 
                    label='Anomalies', color='red', edgecolor='black')
axes[0, 0].set_title('CO2 Emissions Distribution with Anomalies', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('CO2 Emissions (kg)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].legend()

# Distance vs Emissions with anomalies
normal_data = df[~df.index.isin(zscore_anomalies.index)]
axes[0, 1].scatter(normal_data['distance'], normal_data['emissions_co2'], 
                   alpha=0.5, s=20, label='Normal', color='lightblue')
if len(zscore_anomalies) > 0:
    axes[0, 1].scatter(zscore_anomalies['distance'], zscore_anomalies['emissions_co2'], 
                       alpha=0.8, s=50, label='Anomalies', color='red', edgecolor='black', linewidth=1)
axes[0, 1].set_title('Distance vs Emissions (Anomalies Highlighted)', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Distance (km)')
axes[0, 1].set_ylabel('CO2 Emissions (kg)')
axes[0, 1].legend()

# Emissions per km
axes[1, 0].boxplot([df['emissions_per_km']], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7))
axes[1, 0].set_title('Emissions per km Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Emissions per km (kg/km)')
axes[1, 0].grid(True, alpha=0.3)

# Efficiency score (eco_points)
axes[1, 1].hist(df['eco_points'], bins=30, alpha=0.7, color='green', edgecolor='black')
axes[1, 1].axvline(df['eco_points'].mean(), color='red', linestyle='--', 
                    linewidth=2, label=f'Mean: {df["eco_points"].mean():.0f}')
axes[1, 1].set_title('Eco Points Distribution', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('Eco Points')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 3. Isolation Forest Anomaly Detection


In [ ]:
# Prepare features for Isolation Forest
feature_cols = ['weight', 'distance', 'emissions_co2', 'emissions_per_km', 'emissions_per_kg', 'eco_points']
X = df[feature_cols].values

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply Isolation Forest
iso_forest = IsolationForest(contamination=0.1, random_state=42)
df['iso_anomaly'] = iso_forest.fit_predict(X_scaled)
df['iso_anomaly_score'] = iso_forest.score_samples(X_scaled)

# Mark anomalies (Isolation Forest returns -1 for anomalies)
df['is_iso_anomaly'] = df['iso_anomaly'] == -1

print(f"Isolation Forest anomalies detected: {df['is_iso_anomaly'].sum()}")
print(f"\nAnomaly score range: [{df['iso_anomaly_score'].min():.3f}, {df['iso_anomaly_score'].max():.3f}]")
print(f"Mean anomaly score: {df['iso_anomaly_score'].mean():.3f}")


In [ ]:
# Visualize Isolation Forest results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Anomaly scores distribution
axes[0].hist(df[df['is_iso_anomaly'] == False]['iso_anomaly_score'], bins=30, 
             alpha=0.7, label='Normal', color='lightblue')
axes[0].hist(df[df['is_iso_anomaly'] == True]['iso_anomaly_score'], bins=15, 
             alpha=0.8, label='Anomalies', color='red', edgecolor='black')
axes[0].set_title('Isolation Forest Anomaly Scores', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Anomaly Score (lower = more anomalous)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2D visualization: Distance vs Emissions
normal = df[df['is_iso_anomaly'] == False]
anomalies = df[df['is_iso_anomaly'] == True]

axes[1].scatter(normal['distance'], normal['emissions_co2'], alpha=0.5, s=20, 
                label='Normal', color='lightblue')
axes[1].scatter(anomalies['distance'], anomalies['emissions_co2'], alpha=0.8, s=80, 
                label='Anomalies', color='red', edgecolor='black', linewidth=1.5, marker='X')
axes[1].set_title('Isolation Forest Anomaly Detection', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Distance (km)')
axes[1].set_ylabel('CO2 Emissions (kg)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Efficiency Analysis


In [ ]:
# Calculate efficiency metrics
def calculate_efficiency_score(row):
    """Calculate overall efficiency score (0-100)"""
    # Lower emissions per km = higher efficiency
    # Lower emissions per kg = higher efficiency
    # Higher eco points = higher efficiency
    
    # Normalize metrics (assuming max values)
    max_emissions_per_km = 2.0
    max_emissions_per_kg = 0.5
    max_eco_points = 1000
    
    emissions_km_score = max(0, (1 - row['emissions_per_km'] / max_emissions_per_km) * 50)
    emissions_kg_score = max(0, (1 - row['emissions_per_kg'] / max_emissions_per_kg) * 30)
    eco_points_score = (row['eco_points'] / max_eco_points) * 20
    
    return emissions_km_score + emissions_kg_score + eco_points_score

df['efficiency_score'] = df.apply(calculate_efficiency_score, axis=1)

# Categorize efficiency
df['efficiency_category'] = pd.cut(df['efficiency_score'], 
                                    bins=[0, 40, 70, 100], 
                                    labels=['Low', 'Medium', 'High'])

print("Efficiency Distribution:")
print(df['efficiency_category'].value_counts())
print(f"\nAverage Efficiency Score: {df['efficiency_score'].mean():.2f}")
print(f"Efficiency Score Range: [{df['efficiency_score'].min():.2f}, {df['efficiency_score'].max():.2f}]")


In [ ]:
# Efficiency by vehicle type
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Efficiency by vehicle type
vehicle_efficiency = df.groupby('vehicle_type')['efficiency_score'].mean().sort_values(ascending=False)
axes[0, 0].bar(vehicle_efficiency.index, vehicle_efficiency.values, color='teal')
axes[0, 0].set_title('Average Efficiency by Vehicle Type', fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Vehicle Type')
axes[0, 0].set_ylabel('Average Efficiency Score')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Efficiency by fuel type
fuel_efficiency = df.groupby('fuel_type')['efficiency_score'].mean().sort_values(ascending=False)
axes[0, 1].bar(fuel_efficiency.index, fuel_efficiency.values, color='coral')
axes[0, 1].set_title('Average Efficiency by Fuel Type', fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Fuel Type')
axes[0, 1].set_ylabel('Average Efficiency Score')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Efficiency distribution
axes[1, 0].hist(df['efficiency_score'], bins=30, edgecolor='black', alpha=0.7, color='lightgreen')
axes[1, 0].axvline(df['efficiency_score'].mean(), color='red', linestyle='--', 
                   linewidth=2, label=f'Mean: {df["efficiency_score"].mean():.1f}')
axes[1, 0].set_title('Efficiency Score Distribution', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Efficiency Score')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Efficiency vs Emissions
scatter = axes[1, 1].scatter(df['emissions_co2'], df['efficiency_score'], 
                             c=df['distance'], cmap='viridis', alpha=0.6, s=30)
axes[1, 1].set_title('Efficiency vs Emissions (colored by distance)', fontsize=14, fontweight='bold')
axes[1, 1].set_xlabel('CO2 Emissions (kg)')
axes[1, 1].set_ylabel('Efficiency Score')
plt.colorbar(scatter, ax=axes[1, 1], label='Distance (km)')

plt.tight_layout()
plt.show()


## 5. Inefficient Shipment Detection


In [ ]:
# Identify inefficient shipments
inefficient_threshold = df['efficiency_score'].quantile(0.25)  # Bottom 25%
inefficient_shipments = df[df['efficiency_score'] < inefficient_threshold].copy()

print(f"Inefficient shipments detected: {len(inefficient_shipments)} ({len(inefficient_shipments)/len(df)*100:.1f}%)")
print(f"\nEfficiency threshold: {inefficient_threshold:.2f}")

# Analyze inefficient shipments
print("\nInefficient Shipments Analysis:")
print(f"Average emissions: {inefficient_shipments['emissions_co2'].mean():.2f} kg")
print(f"Average distance: {inefficient_shipments['distance'].mean():.2f} km")
print(f"Average weight: {inefficient_shipments['weight'].mean():.2f} kg")
print(f"\nVehicle type distribution:")
print(inefficient_shipments['vehicle_type'].value_counts())
print(f"\nFuel type distribution:")
print(inefficient_shipments['fuel_type'].value_counts())


In [ ]:
# Show top inefficient shipments
print("\nTop 10 Most Inefficient Shipments:")
inefficient_display = inefficient_shipments.nsmallest(10, 'efficiency_score')[
    ['id', 'vehicle_type', 'fuel_type', 'weight', 'distance', 'emissions_co2', 
     'emissions_per_km', 'efficiency_score']
].round(2)
print(inefficient_display.to_string(index=False))


## 6. Route Optimization Opportunities


In [ ]:
# Detect route inefficiencies (unusually long distances for the weight)
df['distance_weight_ratio'] = df['distance'] / df['weight']
df['route_inefficiency'] = df['distance_weight_ratio'] > df['distance_weight_ratio'].quantile(0.9)

route_inefficient = df[df['route_inefficiency']].copy()

print(f"Route inefficiencies detected: {len(route_inefficient)} ({len(route_inefficient)/len(df)*100:.1f}%)")
print("\nRoute Inefficiency Analysis:")
print(f"Average distance: {route_inefficient['distance'].mean():.2f} km")
print(f"Average weight: {route_inefficient['weight'].mean():.2f} kg")
print(f"Average distance/weight ratio: {route_inefficient['distance_weight_ratio'].mean():.4f}")
print(f"\nCompared to overall average ratio: {df['distance_weight_ratio'].mean():.4f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(df[~df['route_inefficiency']]['weight'], 
                df[~df['route_inefficiency']]['distance'], 
                alpha=0.5, s=20, label='Normal', color='lightblue')
axes[0].scatter(route_inefficient['weight'], route_inefficient['distance'], 
                alpha=0.8, s=60, label='Inefficient Routes', color='orange', 
                edgecolor='black', linewidth=1, marker='s')
axes[0].set_title('Route Inefficiency Detection', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Weight (kg)')
axes[0].set_ylabel('Distance (km)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distance/Weight ratio distribution
axes[1].hist(df[~df['route_inefficiency']]['distance_weight_ratio'], bins=30, 
             alpha=0.7, label='Normal', color='lightblue')
axes[1].hist(route_inefficient['distance_weight_ratio'], bins=15, 
             alpha=0.8, label='Inefficient', color='orange', edgecolor='black')
axes[1].set_title('Distance/Weight Ratio Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Distance/Weight Ratio')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## 7. Vehicle Selection Analysis


In [ ]:
# Analyze if correct vehicle type is being used
def recommend_vehicle(weight, distance):
    """Recommend best vehicle type based on weight and distance"""
    if distance < 100:
        if weight < 1000:
            return 'van'
        else:
            return 'truck'
    elif distance < 500:
        if weight < 2000:
            return 'van'
        else:
            return 'truck'
    elif distance < 1000:
        if weight > 3000:
            return 'train'
        else:
            return 'truck'
    else:
        if weight > 2000:
            return 'train'
        elif distance > 2000:
            return 'ship'
        else:
            return 'truck'

df['recommended_vehicle'] = df.apply(lambda x: recommend_vehicle(x['weight'], x['distance']), axis=1)
df['vehicle_mismatch'] = df['vehicle_type'] != df['recommended_vehicle']

mismatches = df[df['vehicle_mismatch']].copy()

print(f"Vehicle selection mismatches: {len(mismatches)} ({len(mismatches)/len(df)*100:.1f}%)")
print("\nMismatch Analysis:")
print(f"Current vehicle types in mismatches:")
print(mismatches['vehicle_type'].value_counts())
print(f"\nRecommended vehicle types:")
print(mismatches['recommended_vehicle'].value_counts())

# Calculate potential savings
mismatches['potential_emissions'] = mismatches.apply(
    lambda x: emission_factors.get((x['recommended_vehicle'], 'electric' if x['recommended_vehicle'] in ['train', 'truck'] else 'diesel'), 0.5) 
    * x['distance'] * (x['weight'] / 1000), axis=1
)
mismatches['emission_savings'] = mismatches['emissions_co2'] - mismatches['potential_emissions']

print(f"\nPotential emission savings: {mismatches['emission_savings'].sum():.2f} kg CO2")
print(f"Average savings per shipment: {mismatches['emission_savings'].mean():.2f} kg CO2")


## 8. Summary and Recommendations


In [ ]:
# Create summary report
print("=" * 70)
print("ANOMALY & EFFICIENCY DETECTION SUMMARY")
print("=" * 70)

print(f"\n1. ANOMALY DETECTION:")
print(f"   - Z-score anomalies: {len(zscore_anomalies)}")
print(f"   - Isolation Forest anomalies: {df['is_iso_anomaly'].sum()}")
print(f"   - True anomalies in data: {df['is_anomaly'].sum()}")

print(f"\n2. EFFICIENCY ANALYSIS:")
print(f"   - Average efficiency score: {df['efficiency_score'].mean():.2f}/100")
print(f"   - Low efficiency shipments: {len(inefficient_shipments)} ({len(inefficient_shipments)/len(df)*100:.1f}%)")
print(f"   - High efficiency shipments: {len(df[df['efficiency_score'] > df['efficiency_score'].quantile(0.75)])} ({len(df[df['efficiency_score'] > df['efficiency_score'].quantile(0.75)])/len(df)*100:.1f}%)")

print(f"\n3. ROUTE OPTIMIZATION:")
print(f"   - Inefficient routes detected: {len(route_inefficient)}")
print(f"   - Average distance for inefficient routes: {route_inefficient['distance'].mean():.2f} km")

print(f"\n4. VEHICLE SELECTION:")
print(f"   - Vehicle mismatches: {len(mismatches)}")
if len(mismatches) > 0:
    print(f"   - Potential emission savings: {mismatches['emission_savings'].sum():.2f} kg CO2")

print(f"\n5. KEY RECOMMENDATIONS:")
print("   ✓ Use electric vehicles for shorter distances (< 200 km)")
print("   ✓ Use trains for heavy cargo over long distances (> 500 km)")
print("   ✓ Optimize routes to minimize unnecessary distance")
print("   ✓ Review vehicle selection for weight-distance combinations")
print("   ✓ Monitor emissions per km as a key efficiency metric")
print("   ✓ Set up alerts for shipments with efficiency score < 40")

print("\n" + "=" * 70)


In [ ]:
# Export results
df_results = df[[
    'id', 'vehicle_type', 'fuel_type', 'weight', 'distance', 
    'emissions_co2', 'emissions_per_km', 'efficiency_score', 
    'efficiency_category', 'is_iso_anomaly', 'route_inefficiency', 
    'vehicle_mismatch', 'recommended_vehicle'
]].copy()

# Save to CSV
df_results.to_csv('anomaly_efficiency_results.csv', index=False)
print("Results exported to 'anomaly_efficiency_results.csv'")

# Save anomaly flags
anomaly_summary = pd.DataFrame({
    'id': df['id'],
    'is_statistical_anomaly': df.index.isin(zscore_anomalies.index),
    'is_isolation_forest_anomaly': df['is_iso_anomaly'],
    'is_inefficient': df['efficiency_score'] < inefficient_threshold,
    'has_route_inefficiency': df['route_inefficiency'],
    'has_vehicle_mismatch': df['vehicle_mismatch']
})

anomaly_summary.to_csv('anomaly_flags.csv', index=False)
print("Anomaly flags exported to 'anomaly_flags.csv'")
